# LAB | Abstractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
# Running LOCALLY (CPU, Python 3.14) instead of Colab:
# the pinned Colab versions below don't exist for Python 3.14, so we use what is installed on this PC
# (torch 2.14 CPU, transformers 5, datasets 5, pinecone 10, sentence-transformers 6). The code is the same.
#
# !pip install -qU torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
# !pip install -qU datasets==2.21.0 huggingface_hub==0.25.2 pinecone-client==3.1.0 sentence-transformers==3.0.1 transformers==4.44.2
import importlib.metadata as md
for package in ["torch", "transformers", "datasets", "pinecone", "sentence-transformers"]:
    print(f"{package:22} {md.version(package)}")

torch                  2.14.1+cpu
transformers           5.18.0
datasets               5.0.1
pinecone               10.2.0
sentence-transformers  6.2.0


**Local only:** this PC tries IPv6 first and waits ~10 s on every connection — prefer IPv4 (safe to run twice).

In [2]:
import socket

if not hasattr(socket, "_original_getaddrinfo"):
    socket._original_getaddrinfo = socket.getaddrinfo
socket.getaddrinfo = lambda *args, **kwargs: sorted(
    socket._original_getaddrinfo(*args, **kwargs), key=lambda addr: addr[0] != socket.AF_INET
)

# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [3]:
from datasets import load_dataset

# We use the "wiki_snippets" dataset, which contains millions of short passages
# extracted from Wikipedia. Each passage comes with useful metadata: the article
# it was taken from (article_title), the section within that article (section_title),
# and the passage's text itself (passage_text). We use the "wiki40b_en_100_0"
# configuration, which contains English passages of ~100 words each.
#
# LOCAL CHANGE (8 GB RAM laptop): streaming all 26 parquet shards used ~2.5 GB of RAM and the kernel was
# killed. Instead we load ONE shard (~0.27 GB download, ~680,000 passages) without streaming: datasets
# keeps it memory-mapped on disk, so it needs almost no RAM — and it is still far more than the 5,000
# History passages we need. Same data, same columns, same seed.
wiki_data = load_dataset(
    'community-datasets/wiki_snippets',
    data_files={'train': 'wiki40b_en_100_0/train-00000-of-00026.parquet'},
    split='train',
).shuffle(seed=960)

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [4]:
# show the contents of a single document in the dataset
wiki_data[0]

{'_id': '{"datasets_id": 118324, "wiki_id": "Q5072996", "sp": 2, "sc": 0, "ep": 6, "ec": 565}',
 'datasets_id': 118324,
 'wiki_id': 'Q5072996',
 'start_paragraph': 2,
 'start_character': 0,
 'end_paragraph': 6,
 'end_character': 565,
 'article_title': 'Chapel Emmanuel Railroad Car',
 'section_title': 'History',
 'passage_text': 'Chapel Emmanuel Railroad Car History The car was being built during the financial panic of 1893. While Barney and Smith was able to build the earlier Baptist chapel car, the Evangel, at cost, it was now a public corporation and was struggling to stay solvent. The price quoted for the car did not include any of the interior necessities. Many items that went into the building of the Emmanuel were donations from corporations: brakes from Westinghouse Air Brake Company, various springs and wheels, along with flatware, blankets and a range for cooking. Still others were donated by the various Baptist'}

In [5]:
# filter only documents with History as section_title
# (startswith also keeps titles like "History and culture")
history = wiki_data.filter(lambda d: d["section_title"].startswith("History"))
print(len(history), "History passages in this shard")

54748 History passages in this shard


Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [6]:
import json
from pathlib import Path
from tqdm.auto import tqdm  # progress bar

# LOCAL CHANGE: 20,000 passages instead of 50,000 — embedding runs on a 2-core CPU, so 50,000 would
# take over an hour here (minutes on a Colab GPU). 5,000 was too few: most questions found no relevant passage.
total_doc_count = 20000
CACHE = Path("history_passages_20000.jsonl")     # ~16 MB: re-runs start right here

counter = 0
docs = []

if CACHE.exists():
    docs = [json.loads(line) for line in CACHE.read_text(encoding="utf-8").splitlines()]
    counter = len(docs)
    print(f"loaded {counter} passages from {CACHE}")
else:
    # iterate through the dataset and apply our filter
    for d in tqdm(history, total=total_doc_count):
        # extract the fields we need - article, section, and passage
        docs.append({
            "article_title": d["article_title"],
            "section_title": d["section_title"],
            "passage_text": d["passage_text"],
        })
        # increase the counter on every iteration
        counter += 1
        # stop once we have enough passages
        if counter == total_doc_count:
            break
    CACHE.write_text("\n".join(json.dumps(doc, ensure_ascii=False) for doc in docs), encoding="utf-8")

loaded 20000 passages from history_passages_20000.jsonl


In [7]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,Chapel Emmanuel Railroad Car,History,Chapel Emmanuel Railroad Car History The car w...
1,Fitzalan Square,History & The Marples tragedy,"this is now a betting shop. However, the best-..."
2,Cuckoo hashing,History & Operation,Cuckoo hashing Cuckoo hashing is a scheme in c...
3,Amtrak's 60 Hz traction power system,History,Amtrak's 60 Hz traction power system Amtrak op...
4,"Oak Bluffs, Massachusetts",History,historic landmarks as well as very expensive r...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [8]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key, 
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager 
# (the key icon in the left sidebar). 
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [9]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [10]:
index_name = "abstractive-question-answering" # choose a meaningful name for your index

In [11]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=768,          # the MPNet retriever outputs 768 numbers per text
        metric="cosine",
        spec=spec,
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=768, total_vector_count=5000, metric='cosine', namespaces=1)

# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [12]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the retriever model from huggingface model hub
retriever = SentenceTransformer("flax-sentence-embeddings/all_datasets_v3_mpnet-base", device=device)
retriever

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [13]:
# we will use batches of 64
batch_size = 64

# LOCAL CHANGE: skip the ~5 min of embedding when the index already holds all passages
if index.describe_index_stats().total_vector_count >= len(df):
    print("index already holds all", len(df), "passages - skipping the upload")
else:
    #You will create embedding for the passage_text variable and be use to include the meta data in each batch
    for i in tqdm(range(0, len(df), batch_size)):
        # find end of batch
        i_end = min(i + batch_size, len(df))
        # extract batch
        batch = df.iloc[i:i_end]
        # generate embeddings for batch
        emb = retriever.encode(batch["passage_text"].tolist()).tolist()
        # get metadata
        meta = batch.to_dict(orient="records")
        # create unique IDs
        ids = [f"{idx}" for idx in range(i, i_end)]
        # add all to upsert list
        to_upsert = list(zip(ids, emb, meta))
        # upsert/insert these records to pinecone
        _ = index.upsert(vectors=to_upsert)
    # wait until Pinecone has counted every vector
    while index.describe_index_stats().total_vector_count < len(df):
        time.sleep(2)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/313 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=768, total_vector_count=20000, metric='cosine', namespaces=1)

# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [53]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
# LOCAL CHANGE: bfloat16 halves the model's RAM (1.6 GB -> 0.8 GB) so it fits on this 8 GB laptop;
# answers stay practically the same. On a GPU you would keep the default float32.
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained(
    'vblagoje/bart_lfqa', dtype=torch.bfloat16 if device == 'cpu' else torch.float32
).to(device)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [54]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode([query]).tolist()[0]
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return xc

In [55]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatinate all context passages
    context = " ".join(context)
    # contcatinate the query and context passages
    query = f"question: {query} context: {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [56]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

QueryResponse(matches=[ScoredVector(id='15244', score=0.663913131, values=[], metadata={'article_title': 'Electrical grid', 'passage_text': "Electrical grid History Early electric energy was produced near the device or service requiring that energy. In the 1880s, electricity competed with steam, hydraulics, and especially coal gas. Coal gas was first produced on customer's premises but later evolved into gasification plants that enjoyed economies of scale. In the industrialized world, cities had networks of piped gas, used for lighting. But gas lamps produced poor light, wasted heat, made rooms hot and smoky, and gave off hydrogen and carbon monoxide.  They also posed a fire hazard. In the 1880s electric lighting soon became advantageous compared to gas lighting.\nElectric utility", 'section_title': 'History'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Sun, 01 Nov 2026 13:37:30 GMT', 'content-type': 'application/json',

In [57]:
from pprint import pprint

In [58]:
# format the query in the form generator expects the input
query = format_query(query, result["matches"])
pprint(query)

('question: when was the first electric power system built? context: <P> '
 'Electrical grid History Early electric energy was produced near the device '
 'or service requiring that energy. In the 1880s, electricity competed with '
 'steam, hydraulics, and especially coal gas. Coal gas was first produced on '
 "customer's premises but later evolved into gasification plants that enjoyed "
 'economies of scale. In the industrialized world, cities had networks of '
 'piped gas, used for lighting. But gas lamps produced poor light, wasted '
 'heat, made rooms hot and smoky, and gave off hydrogen and carbon monoxide.  '
 'They also posed a fire hazard. In the 1880s electric lighting soon became '
 'advantageous compared to gas lighting.\n'
 'Electric utility')


The output looks great. Now let's write a function to generate answers.

In [59]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [60]:
generate_answer(query)

('Electricity was first used for lighting in the late 19th century. In the '
 '1880s, electric lighting was first used for lighting. In the 1890s, electric '
 'lighting was first used for heating')


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [61]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The first wireless message was sent by a telegraph. The telegraph was '
 'invented in 1876 by Alexander Graham Bell and Thomas A. Watson. It was the '
 'first wire conversation ever held.')


To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [62]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

took 23 minutes to connect by manually patching in the route of the call.
On October 9, 1876, Alexander Graham Bell and Thomas A. Watson talked by telephone to each other over a two-mile (3 km) wire stretched between Cambridge and Boston. It was the first wire conversation ever held. Yesterday afternoon the same two men talked by telephone to each other over a 3,400-mile (5,500 km) wire between New York and San Francisco. Dr. Bell, the veteran inventor of the telephone, was in New York, and Mr. Watson, his former associate, was on the other side of the continent. They heard each
---
Free-space optical communication Free-space optical communication (FSO) is an optical communication technology that uses light propagating in free space to wirelessly transmit data for telecommunications or computer networking. "Free space" means air, outer space, vacuum, or something similar. This contrasts with using solids such as optical fiber cable.
The technology is useful where the physical connectio

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [63]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('COVID-19 is a virus that is not a bacterium. It is not a bacterium. It is a '
 'virus that is not a bacterium. It is not a bacterium.')


In [64]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

controversially been cited as a possible cause of the high mortality rate during the 1918 flu pandemic, which killed 50 to 100 million people.
---
1933 when within five weeks in autumn an encephalitis epidemic of explosive proportions broke out in the vicinity of St. Louis, Missouri, and the neighboring St. Louis County.
Over 1,000 cases were reported to the local health departments and the newly constituted National Institutes of Health of the United States was appealed to for epidemiological and investigative expertise.
The previously unknown virus that caused the epidemic was isolated by the NIH team first in monkeys and then in white mice.
---
2019 dengue outbreak in Bangladesh History Dengue fever was first observed in Bangladesh in 2000, claimed 93 deaths in that year. After 3 years the deaths have gradually decreased to almost zero. However it struck again in 2018, killed 26 and affected 10,148 people. The previous highest of affected people prior to 2018 was 6,232 in 2002.
---


Let’s finish with a final few questions.

In [65]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

('The war of currents is a term used to refer to a series of events that '
 'occurred in the early 20th century. The most famous of these events is the '
 'Great Divergence, which occurred in')


In [66]:
query = "who was the firsttwo person on the moon?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

('The first man to walk on the moon was Yuri Gagarin in 1961. He was the first '
 'man to walk on the moon, and the first man to walk on the moon.')


In [67]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

('The Space Shuttle was the most expensive project in the history of NASA. It '
 'cost about $10 billion to build.')


<br>

### 🚀 Bonus Challenges

Try one or more of these to practice:

- Ask your own questions and compare answers using different `top_k` values.
- Ask a question the dataset likely can't answer and observe how the model responds.
- Change the `section_title` filter to a topic other than "History" and rebuild a small index.
- Print the retrieved passages next to each generated answer to see how retrieval quality affects the result.
- Swap the retriever or generator for a different model from Hugging Face and compare outputs.

In [68]:
# Bonus 1 + 4: the same question with different top_k, retrieved passages next to each answer
def ask(question, top_k=5, show_passages=True):
    context = query_pinecone(question, top_k=top_k)
    print(f"Q: {question}   (top_k={top_k})")
    generate_answer(format_query(question, context["matches"]))
    if show_passages:
        for m in context["matches"]:
            print(f"   {m['score']:.2f}  {m['metadata']['article_title']}: {m['metadata']['passage_text'][:110]}...")
    print()

for k in [1, 3, 10]:
    ask("what was the war of currents?", top_k=k)

Q: what was the war of currents?   (top_k=1)
("The war of currents is a very broad term, but it's a very broad one. It "
 "covers a wide range of events, and it's a very broad question. I'll try to "
 'give')
   0.39  Rabat-Salé-Zemmour-Zaer: Salé.  This regime was the headquarters of Barbary pirates and held sway over a turbulent time....

Q: what was the war of currents?   (top_k=3)
("The war of currents is a very broad term, but it's a very broad one. The war "
 "of currents is a very broad term, but it's a very broad one. The war of")
   0.39  Rabat-Salé-Zemmour-Zaer: Salé.  This regime was the headquarters of Barbary pirates and held sway over a turbulent time....
   0.38  Ovče Pole: during the First World War between 14 October and 15 November 1915....
   0.33  Chicago Lawn, Chicago: crash of the Lituanica in 1933....

Q: what was the war of currents?   (top_k=10)
('The war of currents is a term used to refer to a series of events that '
 'occurred in the late 19th century and ea

In [ ]:
# Bonus 2: a question this History index can't answer -> watch the low scores and the made-up answer
ask("What is the best smartphone to buy in 2026?", top_k=3)

Q: What is the best smartphone to buy in 2026?   (top_k=3)


**Observation:** when the best retrieval scores are low (well below the ~0.5–0.7 of good matches), the passages
don't contain the answer, but BART still writes a fluent sentence — that's a **hallucination**. A real system should
check the retrieval score first and answer *"I don't know"* below a threshold (like the IronStore assistant does).

In [31]:
# Bonus: a simple guard — only generate when retrieval found something relevant
MIN_SCORE = 0.45

def safe_answer(question, top_k=5):
    context = query_pinecone(question, top_k=top_k)
    best = context["matches"][0]["score"] if context["matches"] else 0
    if best < MIN_SCORE:
        print(f"Q: {question}\n   I don't know - best retrieval score {best:.2f} is below {MIN_SCORE}\n")
        return
    print(f"Q: {question}   (best score {best:.2f})")
    generate_answer(format_query(question, context["matches"]))
    print()

safe_answer("where did COVID-19 originate?")
safe_answer("How was the first wireless message sent?")

Q: where did COVID-19 originate?   (best score 0.50)
('COVID-19 is a new strain of coronaviruses, which are a family of viruses '
 'that infect humans. The coronavirus family is a family of viruses that '
 'infect humans, and')

Q: How was the first wireless message sent?   (best score 0.50)
('The first wireless message was sent by a telegraph. The telegraph was '
 'invented in 1876 by Alexander Graham Bell and Thomas A. Watson. It was the '
 'first wire conversation ever held.')



<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [32]:
# To delete the Pinecone index, 
# uncomment the line below + run this code cell

# pc.delete_index(index_name)